## Langgraph and multi-agent system

In [2]:
from langgraph.graph import StateGraph, END
from langgraph.graph import MessagesState

### Defining agents as nodes

In [3]:
def researcher(state: MessagesState):
    return {"messagges":f"This is response from researcher agent for the resply of: {state["messages"][-1].content}"}

def writer(state: MessagesState):
    return {"messages":f"This is a complete response from writer on: {state["messages"][-1].content}"}

### Defining the graph

In [6]:
graph = StateGraph(MessagesState)

graph.add_node("researcher", researcher)
graph.add_node("writer", writer)

graph.set_entry_point("researcher")
graph.add_edge("researcher", "writer")
graph.add_edge("writer", END)

### Compiling and invoking the graph

In [8]:
app = graph.compile()

result = app.invoke({"messages":["Write something about emotion."]})
result["messages"][-1].content

'This is a complete response from writer on: Write something about emotion.'

### Using shared memory function

In [1]:
import os
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

from typing import TypedDict, List
from langgraph.graph import StateGraph, END

load_dotenv(override=True)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# Defining a state class
class MyState(TypedDict):
    question : str
    research : str
    answer : str

#Defining the agent-nodes
def researcher(state: MyState):
    q = state["question"]
    research_message = llm.invoke(f"Research this question:{q}").content
    return {"research":research_message}

def writer(state: MyState):
    r = state["research"]
    answer_message = llm.invoke(f"Write a clean answer based on the research:{r}")
    return {"answer":answer_message}

# Building the graph
graph = StateGraph(MyState)

graph.add_node("researcher", researcher)
graph.add_node("writer", writer)

graph.set_entry_point("researcher")
graph.add_edge("researcher","writer")
graph.add_edge("writer",END)

In [3]:
app = graph.compile()

initial_state = MyState()
initial_state["question"] = "Explain why sea-water salty"

response = app.invoke(initial_state)
print(response["answer"].content)

Seawater is primarily salty due to the presence of dissolved salts, with sodium chloride (table salt) being the most prevalent. The salinity of seawater, which averages about 3.5% (or approximately 35 grams of dissolved salts per liter), results from several natural processes:

1. **Weathering of Rocks**: Rainwater, slightly acidic from dissolved carbon dioxide, erodes rocks on land, releasing minerals and ions such as sodium, chloride, calcium, magnesium, and potassium into rivers.

2. **River Transport**: Rivers carry these dissolved minerals and ions to the oceans, continuously contributing to seawater's salinity over time.

3. **Hydrothermal Vents**: Underwater volcanic activity and hydrothermal vents release minerals and metals from the Earth's crust into the ocean, further increasing salt content.

4. **Evaporation**: When seawater evaporates, the water vapor leaves behind salts, raising the concentration of salt in the remaining water, especially in warm, arid regions.

5. **Bio

### Let's start with tools and supervisor agent

In [13]:
# Defining a state class
class MyState(TypedDict):
    question : str
    research : str
    answer : str
    next : str

In [35]:
def calculator_tool(expression : str) -> str:
    try:
        return str(eval(expression))
    except:
        return "Error in evaluating the expression"

# Defining a math agent
def math_agent(state : MyState):
    question = state["question"]
    
    decision = llm.invoke(f"Is this a math expression? Answer only yes or no:{question}")
    print(decision.content)
    if "yes" in decision.content.lower():
        result = calculator_tool(question)
        return {"answer":f"The final answer from math agent is:{result}"}
    else:
        response = llm.invoke(f"Answer this question normally:{question}")
        return {"answer":f"Final answer is:{response.content}"}

In [40]:
#Defining the agent-nodes
def researcher(state: MyState):
    q = state["question"]
    research_message = llm.invoke(f"Research this question:{q}").content
    return {"answer":research_message}

def writer(state: MyState):
    r = state["research"]
    answer_message = llm.invoke(f"Write a clean answer based on the research:{r}")
    return {"answer":answer_message}

In [22]:
# Defining the router agent
def router(state : MyState):
    question = state['question']
    
    decision = llm.invoke(f"""Decide the correct agent for the question.
                          options: math_agent, researcher, writer.
                          Here is the question:{question}
                          """)
    print(decision.content)
    if "math" in decision.content:
        return {"next":"math_agent"}
    elif "research" in decision.content:
        return {"next":"researcher"}
    else:
        return {"next":"writer"}

In [20]:
# Defining route function
def route(state : MyState):
    return f"{state["next"]}"

In [41]:
# Building the graph
graph = StateGraph(MyState)

graph.add_node("Router", router)
graph.add_node("math_agent", math_agent)
graph.add_node("researcher", researcher)
graph.add_node("writer",writer)

graph.set_entry_point("Router")
graph.add_conditional_edges("Router", route, {
    "math_agent":"math_agent",
    "researcher": "researcher",
    "writer": "writer"
})

graph.add_edge("math_agent", END)
graph.add_edge("researcher", END)
graph.add_edge("writer", END)

app = graph.compile()

In [42]:
# Calling the graph with question
first_state = MyState()
first_state["question"] = "tell me about the color of mercury"

response = app.invoke(first_state)
print(response["answer"])

The correct agent for the question "tell me about the color of mercury" is **researcher**.
Mercury, the chemical element with the symbol Hg, is a silvery-white metal at room temperature. It is unique among metals because it is liquid at standard conditions for temperature and pressure. The color of mercury can appear shiny and reflective, similar to that of polished silver. 

When observed in small quantities, mercury can look almost like a mirror, reflecting its surroundings. However, in larger quantities, it may appear darker due to the way light interacts with its surface. Additionally, mercury can form various compounds that may exhibit different colors, but elemental mercury itself is primarily recognized for its metallic, silvery appearance. 

It's important to note that mercury is toxic, and exposure to it can have serious health effects, so it should be handled with care.
